# 02 · Campaign — theozyme → scaffold → metal-aware LigandMPNN (His₃ fixed, Zn context)

**Standard slot:** *design campaign.* **For Project 20 this means:** take the Zn-His₃-OH metal-site
theozyme, scaffold it into a **large pool** of backbones (RFdiffusion2 / Riff-Diff — the **A100**
step; GRACE used ~10k), then **metal-aware LigandMPNN sequence design fixing the three His ligands and
passing the Zn as context** (plus a metal-blind ProteinMPNN baseline for the benchmark), and write a
results CSV (D2).

Runs end-to-end on the **mock** backend with no GPU; switch to the real backends on Colab/HPC.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams
Tools change. Before a campaign, HTTP-check that the pinned upstream repos still exist, and pin the
commit/tag you actually use. **RFdiffusion2, Riff-Diff, and CLEAN are new / fast-moving — VERIFY the
current public release/repo at generation time** (do not assert a repo you are unsure of); the others
below are stable enough to head-check.

In [ ]:
import requests

# Pinned upstreams (pin the COMMIT/TAG you use in env/requirements.txt + LOG.md):
STABLE_UPSTREAMS = {
    "RFdiffusion (classic motif scaffolding)": "https://github.com/RosettaCommons/RFdiffusion",
    "LigandMPNN (metal-aware seq design — CENTRAL)": "https://github.com/dauparas/LigandMPNN",
    "ProteinMPNN (metal-blind baseline)": "https://github.com/dauparas/ProteinMPNN",
    "AutoDock Vina (substrate fit)": "https://github.com/ccsb-scripps/AutoDock-Vina",
    "OpenMM (metal-site MD)": "https://github.com/openmm/openmm",
}
# VERIFY-ONLY (new/fast-moving; confirm the current release before relying on a URL):
VERIFY_UPSTREAMS = [
    "RFdiffusion2 (Dauparas 2025) — VERIFY current public release/repo at generation time",
    "Riff-Diff (Schnettler 2025, Nature) — VERIFY current public release/repo at generation time",
    "CLEAN (CLEAN-style EC/functional classification) — VERIFY current release/repo at generation time",
]

for name, url in STABLE_UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"[{r.status_code}] {name}\n      {url}")
    except Exception as e:
        print(f"[ERR] {name}: {e!r}\n      {url}")
print("\nVERIFY MANUALLY (do not assert a repo URL you are unsure of):")
for v in VERIFY_UPSTREAMS:
    print("  -", v)

## 1 · Build the metal-site theozyme and scaffold it (large pool)
The mock path returns placeholder backbones so the loop runs anywhere. On an A100, switch `METHOD` to
`"rfdiffusion2"` or `"riffdiff"` (verify the release) and `N_SCAFFOLDS` to 1000s–10k.

> **A100 NOTE:** scaffolding a **large pool** (GRACE used ~10k) is the compute bottleneck — and metal-
> site placement is harder than a sidechain motif, so budget extra backbones (many will not hold a
> clean tetrahedral His₃ cage). Free Colab T4 can do a small **RFdiffusion** (classic) metal-motif
> demo (tens of backbones); the real campaign wants an A100 (Colab Pro+) or HPC. The mock backend
> below needs no GPU at all.

In [ ]:
from enzyme_tools import build_theozyme, scaffold_motif

theo = build_theozyme("co2_hydration")

METHOD = "mock"        # -> "rfdiffusion2" | "riffdiff" | "rfdiffusion" on Colab/HPC (verify release)
N_SCAFFOLDS = 12       # -> 1000s-10k for the real campaign (GRACE used ~10k)

scaffolds = scaffold_motif(theo, n=N_SCAFFOLDS, method=METHOD)
print(f"{len(scaffolds)} scaffolds via method={METHOD!r} (mock numbers are SYNTHETIC); "
      f"metal={scaffolds[0]['metal']}")
print("example:", scaffolds[0])

## 2 · Metal-aware LigandMPNN — FIXING the three His ligands (+ a ProteinMPNN baseline)
This is the core of **metal-aware** sequence design: redesign the protein but **keep the three His
ligands fixed** AND pass the **Zn as atom context** so the pocket around the charged metal is designed
correctly. That is why **LigandMPNN**, not vanilla ProteinMPNN, is used. We also generate a
**metal-blind ProteinMPNN** set on the same backbones (His fixed, no metal context) for the
notebook-04 benchmark. On Colab set `TOOL="ligandmpnn"` (CPU-fast) with
`--ligand_mpnn_use_atom_context 1` + the fixed-positions list.

In [ ]:
from enzyme_tools import ligandmpnn_metal, proteinmpnn_metal_blind

TOOL = "mock"          # -> "ligandmpnn" on Colab (CPU-fast), with the Zn atom context
SEQS_PER_BACKBONE = 4

ligands = theo.catalytic_residue_ids()   # the three His ligands to FIX
all_designs = []
for bb in scaffolds:
    # metal-AWARE (the real design path):
    for s in ligandmpnn_metal(bb, ligands, n=SEQS_PER_BACKBONE, tool=TOOL):
        s.update(scaffold_id=bb["design_id"], scaffold_method=bb["method"],
                 motif_rmsd=bb["motif_rmsd"], design_tool="ligandmpnn")
        all_designs.append(s)
    # metal-BLIND baseline (for the benchmark only):
    for s in proteinmpnn_metal_blind(bb, ligands, n=SEQS_PER_BACKBONE, tool=TOOL):
        s.update(scaffold_id=bb["design_id"], scaffold_method=bb["method"],
                 motif_rmsd=bb["motif_rmsd"], design_tool="proteinmpnn")
        all_designs.append(s)

n_lig = sum(d["design_tool"] == "ligandmpnn" for d in all_designs)
n_pm = sum(d["design_tool"] == "proteinmpnn" for d in all_designs)
print(f"{len(all_designs)} sequences total: {n_lig} metal-aware LigandMPNN + {n_pm} metal-blind "
      f"ProteinMPNN ({len(scaffolds)} backbones x {SEQS_PER_BACKBONE} each); His ligands fixed: {ligands}")

## 3 · Predict + score (mock metal-ligand geometry), write the results CSV
On Colab, predict each sequence with AF2/ESMFold, read the **active-site pLDDT**, **add the Zn from
the His₃ geometry** (AF2 does not place metals), and compute the real `metal_ligand_geometry`. Here
the mock backend fills SYNTHETIC values so the CSV — the input to notebook 03 — is produced anywhere.
We give the metal-blind ProteinMPNN arm a systematically worse metal-geometry tendency so the
benchmark in notebook 04 has the expected shape (this is a SYNTHETIC teaching effect).

In [ ]:
import pandas as pd, hashlib
from enzyme_tools import metal_ligand_geometry, dock_substrate, active_site_md

rows = []
site = theo.metal_site
for d in all_designs:
    # On Colab, `pred_pdb` is the AF2-predicted PDB (with the Zn built in) for this design; the mock
    # backend keys off the (non-existent) per-design path so each design gets a DISTINCT SYNTHETIC value.
    pred_pdb = f"results/pred/{d['design_id']}.pdb"
    geo = metal_ligand_geometry(pred_pdb, site)        # mock -> SYNTHETIC (varies per design)
    mlrmsd = geo["metal_ligand_rmsd"]
    # SYNTHETIC teaching effect: penalise the metal-BLIND arm so LigandMPNN looks better at the cage.
    if d["design_tool"] == "proteinmpnn":
        mlrmsd = round(mlrmsd + 0.25, 3)
    sub = theo.substrate.split(";")[0].strip()
    dock = dock_substrate(pred_pdb, sub)               # mock -> SYNTHETIC
    md_res = active_site_md(pred_pdb, ns=10.0)         # mock -> SYNTHETIC (metal-FF caveat applies)
    # SYNTHETIC stand-ins for AF2 confidence + a solubility score so the plumbing runs:
    h = int(hashlib.sha256(d["design_id"].encode()).hexdigest(), 16)
    plddt = 78 + (h % 20)             # 78-97, SYNTHETIC
    plddt_cat = 80 + ((h >> 7) % 18)  # 80-97, SYNTHETIC
    scrmsd = round(0.8 + ((h >> 11) % 200) / 100.0, 2)   # 0.8-2.8, SYNTHETIC
    solubility = round(-1.5 + ((h >> 17) % 300) / 100.0, 2)  # -1.5..1.5, SYNTHETIC (CamSol-style)
    rows.append(dict(
        design_id=d["design_id"], scaffold_id=d["scaffold_id"],
        scaffold_method=d["scaffold_method"], design_tool=d["design_tool"],
        metal_aware=d["metal_aware"], sequence=d["sequence"],
        plddt=plddt, plddt_catalytic=plddt_cat, scrmsd=scrmsd,
        metal_ligand_rmsd=mlrmsd, mean_zn_n_dist=geo["mean_zn_n_dist"],
        mean_n_zn_n_angle=geo["mean_n_zn_n_angle"], solubility=solubility,
        vina_score=dock["vina_score"], md_rmsd=md_res["md_rmsd"], synthetic=True))

camp = pd.DataFrame(rows)
camp.to_csv("results/campaign.csv", index=False)
print("wrote results/campaign.csv", camp.shape, "(ALL NUMBERS SYNTHETIC — mock backend)")
print(f"metal_ligand_rmsd range: {camp['metal_ligand_rmsd'].min()}-{camp['metal_ligand_rmsd'].max()} A "
      f"(target < 0.5); columns include design_tool for the LigandMPNN-vs-ProteinMPNN benchmark")
camp.head()

## D2 checklist
- [ ] Scaffolding run logged (method, release/commit, N backbones — toward ~10k, seed) — A100 for the real campaign.
- [ ] Metal-aware LigandMPNN sequences with the **three His ligands provably fixed** + the **Zn passed as atom context** (logged).
- [ ] Metal-blind **ProteinMPNN baseline** generated on the same backbones (for the benchmark).
- [ ] `results/campaign.csv` with one row per design (real metrics on Colab; mock here), carrying `design_tool` + `metal_ligand_rmsd` + `solubility`.
- [ ] Design log (every config + seed + output path) + 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the shared enzyme filter on `campaign.csv`.